In [2]:
!pip install openai>=1.0.0 groq>=0.4.0 requests>=2.31.0 cohere>=5.0.0
!pip install -q google-genai
!pip install python-dotenv



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from google import genai
import os
from dotenv import load_dotenv
load_dotenv()

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
client = genai.Client(api_key=GOOGLE_API_KEY)
print("Gemini client configured successfully.")

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Hello! Which model are you?"
)

print(response.text)

Gemini client configured successfully.
I am a large language model, trained by Google.


In [ ]:
from groq import Groq

import os
from dotenv import load_dotenv
load_dotenv()

api_key = os.getenv("GROQ_API_KEY") 
client = Groq(api_key=api_key)


def query_groq(prompt: str, model: str = "llama-3.3-70b-versatile", max_tokens: int = 500) -> str:
    """
    Send a prompt to a Groq-hosted model and return the response text.

    Args:
        prompt:     The user message to send.
        model:      The Groq model to use (default: llama-3.3-70b-versatile).
        max_tokens: Maximum tokens in the response.

    Returns:
        The assistant's reply as a string, or an error message.
    """
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": "You are a helpful assistant."},
                {"role": "user",   "content": prompt},
            ],
            max_tokens=max_tokens,
            temperature=0.7,
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        return f"Error: {e}"

if __name__ == "__main__":
    user_prompt = input("Enter your prompt: ")
    print("\nQuerying Groq (Llama)...\n")
    result = query_groq(user_prompt)
    print("Response:")
    print(result)


Querying Groq (Llama)...

Response:
It's nice to meet you. Is there something I can help you with or would you like to chat?


In [ ]:

!sudo apt-get install -y zstd


!curl -fsSL https://ollama.ai/install.sh | sh

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to remove and 37 not upgraded.
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [ ]:
import subprocess, time


proc = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
time.sleep(8)


import requests
try:
    r = requests.get("http://localhost:11434")
    print("✅ Ollama server is running!")
except Exception as e:
    print(f"❌ Server not ready: {e}")

✅ Ollama server is running!


In [ ]:

!ollama pull qwen3.5:9b

In [ ]:
import requests


OLLAMA_BASE_URL = "http://localhost:11434"
DEFAULT_MODEL   = "qwen3.5:9b"        

def query_ollama(prompt: str, model: str = DEFAULT_MODEL) -> str:
    """
    Send a prompt to a locally-running Ollama model and return the response text.

    Args:
        prompt: The user message to send.
        model:  The Ollama model to use.

    Returns:
        The model's reply as a string, or an error message.
    """
    url = f"{OLLAMA_BASE_URL}/api/chat"
    payload = {
        "model":  model,
        "messages": [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user",   "content": prompt},
        ],
        "stream": False,
    }
    try:
        response = requests.post(url, json=payload, timeout=120)
        response.raise_for_status()
        data = response.json()
        return data["message"]["content"].strip()
    except requests.exceptions.ConnectionError:
        return "Error: Cannot connect to Ollama. Make sure the server is running (Cell 1)."
    except Exception as e:
        return f"Error: {e}"

user_prompt = input("Enter your prompt: ")
print(f"\nQuerying Ollama ({DEFAULT_MODEL})...\n")
result = query_ollama(user_prompt)
print("Response:")
print(result)

Enter your prompt: hi

Querying Ollama (qwen3.5:9b)...

Response:
Hello! How are you doing today? Is there anything I can help you with?


In [ ]:
from huggingface_hub import InferenceClient

import os
from dotenv import load_dotenv
load_dotenv()

api_key = os.getenv("HUGGINGFACE_API_KEY")

client = InferenceClient(api_key=api_key)

def query_huggingface(prompt: str, model: str = "MiniMaxAI/MiniMax-M2.5:novita", max_tokens: int = 500) -> str:
    """
    Send a prompt to the Hugging Face Inference API and return the response text.

    Args:
        prompt:     The user message to send.
        model:      The model to use.
        max_tokens: Maximum new tokens to generate.

    Returns:
        The model's reply as a string, or an error message.
    """
    try:
        completion = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": "You are a helpful assistant."},
                {"role": "user",   "content": prompt},
            ],
            max_tokens=max_tokens,
        )
        return completion.choices[0].message.content.strip()
    except Exception as e:
        return f"Error: {e}"


if __name__ == "__main__":
    user_prompt = input("Enter your prompt: ")
    print(f"\nQuerying Hugging Face...\n")
    result = query_huggingface(user_prompt)
    print("Response:")
    print(result)


Querying Hugging Face...

Response:
Hi there! How can I help you today?


In [ ]:
import cohere

import os
from dotenv import load_dotenv
load_dotenv()

api_key = os.getenv("COHERE_API_KEY")

client = cohere.ClientV2(api_key)


def query_cohere(prompt: str, model: str = "command-a-03-2025", max_tokens: int = 500) -> str:
    """
    Send a prompt to a Cohere model and return the response text.

    Args:
        prompt:     The user message to send.
        model:      The Cohere model to use (default: command-a-03-2025).
        max_tokens: Maximum tokens to generate.

    Returns:
        The model's reply as a string, or an error message.
    """
    try:
        response = client.chat(
            model=model,
            messages=[
                {"role": "system", "content": "You are a helpful assistant."},
                {"role": "user",   "content": prompt},
            ],
            max_tokens=max_tokens,
            temperature=0.7,
        )
        return response.message.content[0].text.strip()
    except Exception as e:
        return f"Error: {e}"


if __name__ == "__main__":
    user_prompt = input("Enter your prompt: ")
    print("\nQuerying Cohere...\n")
    result = query_cohere(user_prompt)
    print("Response:")
    print(result)


Querying Cohere...

Response:
Hello! How can I assist you today?
